In [ ]:
!pip install pandas numpy scikit-learn azure-storage-blob tqdm


In [ ]:
#!/usr/bin/env python3
"""
build_strict_mapping_nofallback.py

Fixed version:
- Enforces trailing slashes for directory accuracy.
- Removed internal .json filter from listing to allow .jpg discovery.
- Corrected pathing: Metadata in training/ and Images in training/training/Images/.
"""

import os
import json
import csv
from azure.storage.blob import BlobServiceClient
from tqdm import tqdm
from os.path import basename, splitext
import re

# --------- CONFIG ----------
conn_str = ""

if not conn_str:
    raise RuntimeError("Please set AZURE_STORAGE_CONN_STR in environment.")

# Path logic based on your structure:
prefix = "training"
image_parent = "training/training"  # The nested structure you mentioned
target_folder = "Images"
remote_meta_folder = "RandomSamplingMetadata"
container_name = "london"
angles = [0, 90, 180, 270]

output_csv = "strict_mapping_nofallback.csv"
# ---------------------------

svc = BlobServiceClient.from_connection_string(conn_str)
container_client = svc.get_container_client(container_name)

def extract_year(meta_json, image_blob_name):
    """Try metadata first (common keys), then fall back to searching the image filename for a 4-digit year."""
    # check common metadata keys
    for k in ("year", "capture_year", "captureDate", "date", "capture_date", "timestamp"):
        v = meta_json.get(k) if isinstance(meta_json, dict) else None
        if v is not None:
            m = re.search(r'(19|20)\d{2}', str(v))
            if m:
                return m.group(0)
    # fallback: search image blob name
    m = re.search(r'(19|20)\d{2}', image_blob_name or "")
    return m.group(0) if m else ""

def list_blob_names(prefix_filter):
    """Returns list of ALL blob names under given prefix_filter."""
    if prefix_filter and not prefix_filter.endswith("/"):
        prefix_filter += "/"
    return [b.name for b in container_client.list_blobs(name_starts_with=prefix_filter)]

def download_json_blob(blob_name):
    return json.loads(container_client.get_blob_client(blob_name).download_blob().readall())

def find_lat_lng(obj):
    lat = lng = None
    lat_keys = {"lat", "latitude", "y"}
    lng_keys = {"lng", "lon", "lngt", "longitude", "x"}
    def is_lat(v):
        try: return isinstance(v, (int, float)) and -90 <= float(v) <= 90
        except: return False
    def is_lng(v):
        try: return isinstance(v, (int, float)) and -180 <= float(v) <= 180
        except: return False

    def _r(o):
        nonlocal lat, lng
        if lat is not None and lng is not None: return
        if isinstance(o, dict):
            low = {k.lower(): k for k in o.keys()}
            for lk in lat_keys:
                if lk in low:
                    for pk in lng_keys:
                        if pk in low:
                            lv = o[low[lk]]; rv = o[low[pk]]
                            if is_lat(lv) and is_lng(rv):
                                lat, lng = float(lv), float(rv); return
            for v in o.values(): _r(v)
        elif isinstance(o, list):
            for it in o: _r(it)
    _r(obj)
    return lat, lng

# 1. Get Metadata Blobs from training/RandomSamplingMetadata/
meta_path = f"{prefix}/{remote_meta_folder}"
print(f"Listing metadata from: {meta_path}/")
meta_blobs = [b for b in list_blob_names(meta_path) if b.endswith(".json")]
print(f"Found {len(meta_blobs)} metadata JSON blobs.")

# 2. Get Image Blobs from training/training/Images/
image_path = f"{image_parent}/{target_folder}"
print(f"Listing images from: {image_path}/")
image_blob_set = set(list_blob_names(image_path))
print(f"Total items in image folder: {len(image_blob_set)}")

# 3. Process
rows = []
for meta_blob_name in tqdm(sorted(meta_blobs), desc="Processing metadata"):
    try:
        meta_json = download_json_blob(meta_blob_name)
    except Exception as e:
        rows.append({"metadata_blob_name": meta_blob_name, "image_blob_name": "", "lat": "", "lng": "", "year": ""})
        continue

    meta_basename = splitext(basename(meta_blob_name))[0]
    seq = meta_json.get("seq") or meta_json.get("id") or meta_basename
    lat, lng = find_lat_lng(meta_json)

    exact_matches = []

    # Check angled images: training/training/Images/{seq}_{h}.jpg
    for h in angles:
        expected = f"{image_parent}/{target_folder}/{seq}_{h}.jpg"
        if expected in image_blob_set:
            exact_matches.append(expected)

    # Check plain images: training/training/Images/{seq}.jpg
    expected_plain = f"{image_parent}/{target_folder}/{seq}.jpg"
    if expected_plain in image_blob_set and expected_plain not in exact_matches:
        exact_matches.append(expected_plain)

    if exact_matches:
        for img in exact_matches:
            rows.append({
                "metadata_blob_name": meta_blob_name,
                "image_blob_name": img,
                "lat": lat if lat is not None else "",
                "lng": lng if lng is not None else "",
                "year": extract_year(meta_json, img)
            })
    else:
        rows.append({
            "metadata_blob_name": meta_blob_name,
            "image_blob_name": "",
            "lat": lat if lat is not None else "",
            "lng": lng if lng is not None else "",
            "year": extract_year(meta_json, "")
        })

# Write CSV
with open(output_csv, "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=["metadata_blob_name", "image_blob_name", "lat", "lng", "year"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Successfully wrote {len(rows)} rows to {output_csv}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')